
# Bengali NLP Safety — V23 Conditional Attribute BanglaBERT

**Protected Kaggle champion: V14 = 0.586**

Small threshold/rule changes have plateaued. V23 uses a different formulation of the task.

In the labeled data, all six secondary attributes occur only inside toxic examples:

- `implicit_toxicity`
- `sarcasm`
- `contains_profanity`
- `threatening`
- `identity_attack`
- `sexual_content`

So instead of asking one model to separate rare attributes from thousands of easy non-toxic negatives,
V23 trains **BanglaBERT only on `is_toxic=1` examples** to estimate:

`P(attribute | toxic, text)`

At inference we combine that conditional probability with the proven V14 toxic probability and then
blend conservatively with the current V14 attribute probability.

### Why this is different from V15

V15 trained rare labels over the full dataset. V23 removes easy non-toxic examples from the specialist
training objective and makes the model learn distinctions **within toxic language**.

### Safety

- 5 grouped OOF folds.
- Validation rows are always OOF.
- V14 probabilities are reconstructed exactly.
- Per-label candidates must pass a fold-stability gate.
- V14 remains unchanged if nothing passes.

## Run

1. Open in Google Colab.
2. Set **T4 GPU**.
3. Run cell 1 and upload **`v23_data.zip`**.
4. Run the rest.
5. At the end download **`v23_results.zip`**.
6. Send that ZIP to ChatGPT **before submitting a V23 CSV to Kaggle**.


In [ ]:

# 1. Upload data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v23_data.zip" in uploaded, "Upload the supplied v23_data.zip."

DATA_DIR = pathlib.Path("/content/v23_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v23_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Dependencies + BanglaBERT preflight
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

from transformers import AutoConfig, AutoTokenizer

MODEL_NAME = "csebuetnlp/banglabert"
_ = AutoConfig.from_pretrained(MODEL_NAME)
_ = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ BanglaBERT preflight passed.")


In [ ]:

# 3. Settings, imports, data
import gc, re, math, random, json, zipfile, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

SEED = 42
N_FOLDS = 5
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
EPOCHS = 4
SNAPSHOT_EPOCHS = [3,4]
TRAIN_BATCH_SIZE = 8
ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 32
LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

ALL_LABELS = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
ATTRS = ALL_LABELS[1:]

# Attribute replacement gate.
MIN_GAIN = 0.008
MIN_WINS = 3
MAX_LOSSES = 1

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable T4 GPU."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2_valid = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2_test = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
valid_folds = pd.read_csv(DATA_DIR/"validation_folds.csv", dtype={"ID":str})
champion = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})

with np.load(DATA_DIR/"old_banglabert_predictions.npz") as z:
    old_bang_valid=z["valid"].copy()
    old_bang_test=z["test"].copy()

with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z:
    old_xl_valid=z["valid"].copy()
    old_xl_test=z["test"].copy()

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert champion["ID"].tolist()==test["ID"].tolist()
assert v2_valid["ID"].tolist()==valid["ID"].tolist()
assert v2_test["ID"].tolist()==test["ID"].tolist()

combined = pd.concat([train,valid],ignore_index=True)

def clean_text(x):
    x = unicodedata.normalize("NFC",str(x))
    x = x.replace("\u200c"," ").replace("\u200d"," ")
    return re.sub(r"\s+"," ",x).strip()

combined["clean_text"]=combined["text"].map(clean_text)
test["clean_text"]=test["text"].map(clean_text)

# Verify the hierarchy in the labeled data.
violations = {
    a:int(((combined[a]==1)&(combined["is_toxic"]==0)).sum())
    for a in ATTRS
}
print("Attribute-positive / non-toxic violations:",violations)
assert sum(violations.values())==0

print("Combined toxic rows:",int(combined["is_toxic"].sum()),"/",len(combined))
print("Attribute positives among toxic rows:")
display(combined.loc[combined.is_toxic==1,ATTRS].sum().to_frame("positives"))


In [ ]:

# 4. Balanced grouped 5-fold assignment
def normalize_group_text(x):
    x=unicodedata.normalize("NFKC",str(x))
    return re.sub(r"\s+"," ",x).strip().lower()

combined["group_text"]=combined["text"].map(normalize_group_text)

gs=combined.groupby("group_text",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in ALL_LABELS}
).reset_index()

gp=np.array([combined[l].sum() for l in ALL_LABELS],float)
total=float(len(combined))
rarity=1/np.sqrt(np.maximum(gp,1))
mat=gs[[f"sum_{l}" for l in ALL_LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+.005*gs["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(gs))*1e-6))

fp=np.zeros((N_FOLDS,len(ALL_LABELS))); fs=np.zeros(N_FOLDS); assign={}

for idx in order:
    row=gs.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in ALL_LABELS],float)
    size=float(row["size"])
    costs=[]
    for f in range(N_FOLDS):
        p=fp.copy(); s=fs.copy()
        p[f]+=vec; s[f]+=size
        costs.append(
            np.mean((p/np.maximum(gp[None,:],1)-1/N_FOLDS)**2)
            + .35*np.mean((s/total-1/N_FOLDS)**2)
        )
    best=int(np.argmin(costs))
    assign[row["group_text"]]=best
    fp[best]+=vec; fs[best]+=size

combined["fold"]=combined["group_text"].map(assign).astype(int)

counts=combined.groupby("fold").size().reindex(range(N_FOLDS),fill_value=0)
tox_counts=combined.groupby("fold")["is_toxic"].sum().reindex(range(N_FOLDS),fill_value=0)
pos_table=combined.groupby("fold")[ATTRS].sum().reindex(range(N_FOLDS),fill_value=0)

assert (counts>0).all()
assert (tox_counts>0).all()
assert combined.groupby("group_text")["fold"].nunique().max()==1

print("✅ fold sizes:",counts.to_dict())
print("toxic rows by fold:",tox_counts.to_dict())
display(pos_table)


In [ ]:

# 5. Dataset helpers
class AttrDataset(Dataset):
    def __init__(self,frame,tokenizer,with_labels=True):
        self.texts=frame["clean_text"].tolist()
        self.targets=frame[ATTRS].to_numpy(np.float32) if with_labels else None
        self.tokenizer=tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self,i):
        enc=self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.targets is not None:
            enc["labels"]=self.targets[i]
        return enc

class Collator:
    def __init__(self,tok):
        self.tok=tok
    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(np.stack([x.pop("labels") for x in items]),dtype=torch.float32)
        batch=self.tok.pad(items,padding=True,pad_to_multiple_of=8,return_tensors="pt")
        if y is not None:
            batch["labels"]=y
        return batch

@torch.no_grad()
def predict(model,loader):
    assert len(loader.dataset)>0
    model.eval(); parts=[]
    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp).logits
        parts.append(torch.sigmoid(logits.float()).cpu().numpy())
    out=np.concatenate(parts)
    assert np.isfinite(out).all()
    return out


In [ ]:

# 6. Train conditional 5-fold BanglaBERT specialist
RUN_DIR=Path("/content/v23_output")
RUN_DIR.mkdir(exist_ok=True)

# OOF predictions for ALL combined rows.
oof=np.zeros((len(combined),len(ATTRS)),np.float32)
test_fold_probs=[]
histories=[]
fold_seeds=[42,3407,2026,17,1337]
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

for fold in range(N_FOLDS):
    # IMPORTANT: training uses only toxic examples from non-held folds.
    tr=combined[(combined.fold!=fold)&(combined.is_toxic==1)].copy()
    # Predict every example in the held fold, toxic and non-toxic.
    va=combined[combined.fold==fold].copy()

    assert len(tr)>0 and len(va)>0
    assert set(tr["group_text"]).isdisjoint(set(va["group_text"]))

    cache=RUN_DIR/f"conditional_fold_{fold}.npz"
    hist_path=RUN_DIR/f"conditional_fold_{fold}_history.csv"

    if cache.exists():
        with np.load(cache,allow_pickle=False) as z:
            idx=z["va_index"]; vp=z["valid"]; tp=z["test"]
        if np.array_equal(idx,va.index.to_numpy()) and vp.shape==(len(va),len(ATTRS)) and tp.shape==(len(test),len(ATTRS)):
            print("✅ reuse fold",fold)
            oof[idx]=vp
            test_fold_probs.append(tp)
            if hist_path.exists():
                histories.extend(pd.read_csv(hist_path).to_dict("records"))
            continue

    print("\n"+"="*70)
    print(f"fold {fold}: conditional train={len(tr)} | held all rows={len(va)}")
    seed_everything(fold_seeds[fold])

    col=Collator(tokenizer)
    tl=DataLoader(AttrDataset(tr,tokenizer,True),batch_size=TRAIN_BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True,collate_fn=col)
    vl=DataLoader(AttrDataset(va,tokenizer,True),batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=col)
    el=DataLoader(AttrDataset(test,tokenizer,False),batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=col)

    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(ATTRS),
        problem_type="multi_label_classification",
    ).to("cuda")

    pos=tr[ATTRS].sum().to_numpy(float)
    neg=len(tr)-pos

    # Conditional imbalance is much smaller than full-data imbalance.
    pos_weight=np.sqrt(neg/np.maximum(pos,1))
    pos_weight=np.clip(pos_weight,.75,4.0)
    print("conditional pos_weight:",dict(zip(ATTRS,np.round(pos_weight,3))))
    loss_fn=torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight,dtype=torch.float32,device="cuda")
    )

    no_decay=["bias","LayerNorm.weight","layer_norm.weight"]
    groups=[
        {"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],"weight_decay":WEIGHT_DECAY},
        {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],"weight_decay":0.0},
    ]
    optimizer=torch.optim.AdamW(groups,lr=LR)
    steps=math.ceil(len(tl)/ACCUM_STEPS)*EPOCHS
    scheduler=get_linear_schedule_with_warmup(
        optimizer,int(steps*WARMUP_RATIO),steps
    )
    scaler=torch.amp.GradScaler("cuda")
    vs=[]; ts=[]; fh=[]

    for epoch in range(1,EPOCHS+1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        run=0.0; seen=0

        for step,b in enumerate(tqdm(tl,desc=f"conditional f{fold} e{epoch}/{EPOCHS}")):
            y=b.pop("labels").to("cuda",non_blocking=True)
            inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inp).logits

            loss=loss_fn(logits.float(),y)
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss")

            scaler.scale(loss/ACCUM_STEPS).backward()
            run+=float(loss.item())*len(y); seen+=len(y)

            if (step+1)%ACCUM_STEPS==0 or step+1==len(tl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                old=scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale()>=old:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        rec={"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}
        print(rec); fh.append(rec)
        pd.DataFrame(fh).to_csv(hist_path,index=False)

        if epoch in SNAPSHOT_EPOCHS:
            vs.append(predict(model,vl))
            ts.append(predict(model,el))

    vp=np.mean(vs,axis=0)
    tp=np.mean(ts,axis=0)

    oof[va.index.to_numpy()]=vp
    test_fold_probs.append(tp)
    histories.extend(fh)
    np.savez_compressed(cache,va_index=va.index.to_numpy(),valid=vp,test=tp)

    del model,tl,vl,el,optimizer,scheduler,scaler
    gc.collect(); torch.cuda.empty_cache()

assert len(test_fold_probs)==N_FOLDS
conditional_test=np.mean(test_fold_probs,axis=0)

np.savez_compressed(
    RUN_DIR/"conditional_oof_test.npz",
    oof=oof,
    test=conditional_test
)
pd.DataFrame(histories).to_csv(RUN_DIR/"conditional_training_history.csv",index=False)
print("✅ conditional OOF complete.")


In [ ]:

# 7. Reconstruct exact V14 probability family
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))
def sigmoid(z):
    return 1/(1+np.exp(-z))

li={l:i for i,l in enumerate(ALL_LABELS)}
base_valid={}
base_test={}

base_valid["is_toxic"]=.875*v2_valid["is_toxic_prob"].to_numpy(float)+.125*old_xl_valid[1:3,:,0].mean(0)
base_test["is_toxic"]=.875*v2_test["is_toxic_prob"].to_numpy(float)+.125*old_xl_test[1:3,:,0].mean(0)

j=li["implicit_toxicity"]
base_valid["implicit_toxicity"]=sigmoid(.9*logit(v2_valid["implicit_toxicity_prob"].to_numpy(float))+.1*logit(old_bang_valid.mean(0)[:,j]))
base_test["implicit_toxicity"]=sigmoid(.9*logit(v2_test["implicit_toxicity_prob"].to_numpy(float))+.1*logit(old_bang_test.mean(0)[:,j]))

base_valid["sarcasm"]=v2_valid["sarcasm_prob"].to_numpy(float)
base_test["sarcasm"]=v2_test["sarcasm_prob"].to_numpy(float)

j=li["contains_profanity"]
base_valid["contains_profanity"]=.95*v2_valid["contains_profanity_prob"].to_numpy(float)+.05*old_bang_valid[0,:,j]
base_test["contains_profanity"]=.95*v2_test["contains_profanity_prob"].to_numpy(float)+.05*old_bang_test[0,:,j]

j=li["threatening"]
base_valid["threatening"]=.95*v2_valid["threatening_prob"].to_numpy(float)+.05*old_xl_valid[0,:,j]
base_test["threatening"]=.95*v2_test["threatening_prob"].to_numpy(float)+.05*old_xl_test[0,:,j]

j=li["identity_attack"]
base_valid["identity_attack"]=.7*v2_valid["identity_attack_prob"].to_numpy(float)+.3*old_bang_valid[2,:,j]
base_test["identity_attack"]=.7*v2_test["identity_attack_prob"].to_numpy(float)+.3*old_bang_test[2,:,j]

base_valid["sexual_content"]=v2_valid["sexual_content_prob"].to_numpy(float)
base_test["sexual_content"]=v2_test["sexual_content_prob"].to_numpy(float)

historical_thresholds={
    "is_toxic":0.6850110311061144,
    "implicit_toxicity":0.1416,
    "sarcasm":0.175,
    "contains_profanity":0.245,
    "threatening":0.16558668018085876,
    "identity_attack":0.3814,
    "sexual_content":0.245,
}

rebuilt=champion.copy()
for l in ALL_LABELS:
    rebuilt[l]=(base_test[l]>=historical_thresholds[l]).astype(np.int8)

diffs={l:int((rebuilt[l].to_numpy()!=champion[l].to_numpy()).sum()) for l in ALL_LABELS}
print("V14 reconstruction:",diffs)
assert sum(diffs.values())==0
print("✅ exact V14 reconstructed.")


In [ ]:

# 8. Prepare conditional OOF signals on original validation
orig_idx=np.arange(len(train),len(train)+len(valid))
cond_valid=oof[orig_idx]

fold_map=valid_folds.set_index("ID")["fold"]
selection_folds=valid["ID"].map(fold_map).to_numpy(int)
assert set(selection_folds)==set(range(N_FOLDS))

y_valid=valid[ATTRS].to_numpy(np.int8)
tox_valid=base_valid["is_toxic"]
tox_test=base_test["is_toxic"]

# Four interpretations of P(attribute | toxic).
hier_valid={}
hier_test={}

for j,a in enumerate(ATTRS):
    cval=cond_valid[:,j]
    ctest=conditional_test[:,j]

    hier_valid[(a,"conditional_raw")]=cval
    hier_test[(a,"conditional_raw")]=ctest

    hier_valid[(a,"tox_sqrt")]=cval*np.sqrt(tox_valid)
    hier_test[(a,"tox_sqrt")]=ctest*np.sqrt(tox_test)

    hier_valid[(a,"tox_product")]=cval*tox_valid
    hier_test[(a,"tox_product")]=ctest*tox_test

    hier_valid[(a,"tox_square")]=cval*(tox_valid**2)
    hier_test[(a,"tox_square")]=ctest*(tox_test**2)

print("Prepared hierarchical signals.")


In [ ]:

# 9. Cross-fitted positive-F1 threshold selection
def f1_pos(y,pred):
    return float(f1_score(y,pred,zero_division=0))

def best_threshold(y,p):
    vals=np.unique(np.asarray(p,float))
    if len(vals)>600:
        vals=np.quantile(vals,np.linspace(0,1,600))
    cand=np.unique(np.r_[0.,vals,1.000001])
    scores=np.array([f1_pos(y,p>=t) for t in cand])
    best=scores.max()
    tied=cand[np.isclose(scores,best,atol=1e-12,rtol=0)]
    return float(np.median(tied))

def crossfit(y,p):
    pred=np.zeros(len(y),dtype=np.int8)
    fs=[]; ts=[]
    for fold in range(N_FOLDS):
        held=selection_folds==fold
        t=best_threshold(y[~held],p[~held])
        ts.append(t)
        pred[held]=(p[held]>=t).astype(np.int8)
        fs.append(f1_pos(y[held],pred[held]))
    return {
        "score":f1_pos(y,pred),
        "fold_scores":np.asarray(fs,float),
        "thresholds":np.asarray(ts,float),
        "prediction":pred,
    }

baseline_cv={}
for j,a in enumerate(ATTRS):
    baseline_cv[a]=crossfit(y_valid[:,j],base_valid[a])

display(pd.DataFrame([
    {
        "label":a,
        "baseline_crossfit":baseline_cv[a]["score"],
        "fold_scores":np.round(baseline_cv[a]["fold_scores"],4).tolist()
    }
    for a in ATTRS
]))


In [ ]:

# 10. Search conservative conditional blends
def blend(base,new,w,mode):
    if mode=="prob":
        return (1-w)*base+w*new
    return sigmoid((1-w)*logit(base)+w*logit(new))

search_rows=[]
best_by_attr={}

for j,a in enumerate(ATTRS):
    y=y_valid[:,j]
    b=base_valid[a]
    bcv=baseline_cv[a]

    best=None

    for signal in ["conditional_raw","tox_sqrt","tox_product","tox_square"]:
        h=hier_valid[(a,signal)]

        for mode in ["prob","logit"]:
            for wi in range(1,13): # 0.05..0.60
                w=wi/20
                p=blend(b,h,w,mode)
                cv=crossfit(y,p)

                wins=int((cv["fold_scores"]>bcv["fold_scores"]+1e-12).sum())
                losses=int((cv["fold_scores"]<bcv["fold_scores"]-1e-12).sum())
                ties=N_FOLDS-wins-losses
                gain=cv["score"]-bcv["score"]
                passes=(
                    gain>=MIN_GAIN and
                    wins>=MIN_WINS and
                    losses<=MAX_LOSSES
                )

                rec={
                    "label":a,
                    "signal":signal,
                    "mode":mode,
                    "weight":w,
                    "score":cv["score"],
                    "baseline":bcv["score"],
                    "gain":gain,
                    "wins":wins,
                    "ties":ties,
                    "losses":losses,
                    "passes_gate":passes,
                    "median_threshold":float(np.median(cv["thresholds"])),
                }
                search_rows.append(rec)

                # Favor stable improvement with smaller intervention.
                key=(passes,cv["score"],wins,-losses,-w)
                if best is None or key>best["key"]:
                    best={"key":key,"rec":rec,"cv":cv}

    best_by_attr[a]=best

search=pd.DataFrame(search_rows).sort_values(
    ["passes_gate","gain","wins","losses"],
    ascending=[False,False,False,True]
)
search.to_csv(RUN_DIR/"v23_blend_search.csv",index=False)

summary=[]
for a,b in best_by_attr.items():
    r=b["rec"].copy()
    r["weighted_final_score_gain"]=r["gain"]/12
    summary.append(r)

summary=pd.DataFrame(summary).sort_values(
    "weighted_final_score_gain",ascending=False
)
summary.to_csv(RUN_DIR/"v23_selection_summary.csv",index=False)

display(summary.round(5))


In [ ]:

# 11. Build controlled submissions
def test_prob_for(a,rec):
    h=hier_test[(a,rec["signal"])]
    return blend(
        base_test[a],
        h,
        float(rec["weight"]),
        rec["mode"]
    )

passed=summary[summary["passes_gate"]==True].copy()

def make_submission(attr_list):
    out=champion.copy()
    details={}
    for a in attr_list:
        b=best_by_attr[a]
        rec=b["rec"]
        p=test_prob_for(a,rec)
        threshold=float(np.median(b["cv"]["thresholds"]))
        pred=(p>=threshold).astype(np.int8)
        before=out[a].to_numpy(int)
        out[a]=pred

        details[a]={
            "signal":rec["signal"],
            "mode":rec["mode"],
            "weight":float(rec["weight"]),
            "threshold":threshold,
            "gain":float(rec["gain"]),
            "wins":int(rec["wins"]),
            "ties":int(rec["ties"]),
            "losses":int(rec["losses"]),
            "changed_rows":int((before!=pred).sum()),
            "positives_before":int(before.sum()),
            "positives_after":int(pred.sum()),
        }

    assert out["ID"].tolist()==champion["ID"].tolist()
    assert out[ALL_LABELS].isin([0,1]).all().all()
    return out,details

best_single=[passed.iloc[0]["label"]] if len(passed) else []
all_passed=passed["label"].tolist() if len(passed) else []

single,single_details=make_submission(best_single)
allp,all_details=make_submission(all_passed)

single.to_csv(RUN_DIR/"submission_v23_best_single.csv",index=False)
allp.to_csv(RUN_DIR/"submission_v23_all_passed.csv",index=False)

report={
    "version":"v23_conditional_attribute_banglabert",
    "champion_score":0.586,
    "best_single":best_single,
    "all_passed":all_passed,
    "best_single_details":single_details,
    "all_passed_details":all_details,
    "gate":{
        "min_gain":MIN_GAIN,
        "min_wins":MIN_WINS,
        "max_losses":MAX_LOSSES,
    },
    "warning":"Send v23_results.zip to ChatGPT before Kaggle submission."
}
(RUN_DIR/"v23_report.json").write_text(json.dumps(report,indent=2))

print("\nRECOMMENDATION")
if best_single:
    print("Candidate passed. Best single:",best_single[0])
    print("Send v23_results.zip to ChatGPT before submitting.")
else:
    print("No conditional attribute candidate passed. Keep V14 = 0.586.")

print("All passed:",all_passed)


In [ ]:

# 12. Package results
zip_path=Path("/content/v23_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)
from google.colab import files
files.download(str(zip_path))
